# Anchoring regression V4 (PFC / mFC dataset) - past *and* future lagsMirror of [`LEC_elasticnet_regression_v4.ipynb`](../../code/LEC_elasticnet_regression_v4.ipynb)on the mFC dataset. The analysis module[`elasticnet_regression_v4.py`](elasticnet_regression_v4.py) is kept **byte-identical** to theLEC copy (`diff` it); only the loader cells below and the grouping column differ.**This dataset is El-Gaby's own.** `data/MetaData/combined_ABCDonly_days.npy` is the file`Figure5_Regression.ipynb` itself loads, and the 25 recdays are `me08/me10/me11/ah03/ah04/ah07/ab03`. So this is not "the same analysis on another region" - it re-runs the published Figure 5analysis on the published data, and each caveat below is a statement about the paper.**What was measured on this dataset while planning** (LEC in brackets):| | PFC | LEC ||---|---|---|| phase transitions deviating from +1 mod 3 | **0 / 59,904** | 0 / 39,732 || `alpha=0.01` all-zero fits | **57%** | 60% || median per-neuron `alpha_max` | **0.00871** | 0.00708 || leg longest:shortest, median | **1.82x** (p90 2.91x, max 25.15x) | 2.26x || sessions with leg ratio >= 2x | **36%** (>=3x: 9%) | - || untracked-bin bug | **absent** (real NaN) | 4.1% kept as rows || units / recdays | **1252 / 25** | 2851 / 25 |* The **108/324 structural fact** and the **exactly-zero-off-preferred-phase prediction**  transfer exactly - the phase cycle is as strict here as in LEC.* The **alpha problem transfers essentially unchanged** despite PFC firing faster (3.79 Hz vs  2.94 Hz): `alpha_max` scales as 1/n and PFC sessions are longer, so the two cancel.* The **leg-duration confound on the state-tuning filter is milder but present** - 1.82x sits  between the measured 1.5x (FPR 0.41) and 2.0x (FPR 0.95) rows.* `drop_untracked_bins` is a **no-op** here: that bug came from LEC's `locs_to_int` mapping  SLEAP NaN to 0. PFC `Location_raw` uses real NaN.**PFC has no anatomy** - no `unit_regions`, no `anatomy_split`. `build_unit_table` thereforegroups by `mouse`; every regression column is identical to the LEC path.Gate: `python elasticnet_v4_synthetics.py` - 28 controls, including exact reproduction of v3.

In [ ]:
import numpy as npimport scipy.stats as stfrom scipy import statsfrom scipy.stats import zscorefrom scipy.ndimage import gaussian_filter1dimport matplotlib.pyplot as pltimport seaborn as snsimport pandas as pdimport os, picklefrom tqdm import tqdm

In [ ]:
DATA_FOLDER = '/ceph/behrens/adam_harris/Taskspace_abstraction_lEC/mFC_data/data'META = os.path.join(DATA_FOLDER, 'MetaData')

In [ ]:
# The canonical recday list -- the same file `Figure5_Regression.ipynb` loads.mouse_recdays = list(np.load(os.path.join(META, 'combined_ABCDonly_days.npy')).astype(str))print(f'{len(mouse_recdays)} recdays in combined_ABCDonly_days.npy')print('first 3:', mouse_recdays[:3])

In [ ]:
# Build the LEC-shaped data_dic from the PFC directory layout.# compute_norm=False: V4 works entirely in raw time and never touches Neurons_norm, so the# ~4 GB of normalised arrays the v2 path needed are pure overhead here.from glm_analysis_v2 import build_data_dic_from_pfcdata_dic = build_data_dic_from_pfc(DATA_FOLDER, mouse_recdays, compute_norm=False)mouse_recdays = sorted(data_dic.keys())print(f'\n{len(mouse_recdays)} recdays loaded')

In [ ]:
# Session selection, in two steps.## 1. Keep one session per unique task. This is exactly El-Gaby's `non_repeat_ses_maker`:#    both are dedup by exact array equality of the reward sequence.# 2. Apply his one hand-exclusion. me11 session 3's task [7,4,3,8] shares 3 of 4 goals with#    session 0's [7,4,3,5] -- his comment is "almost identical to session 0 (mistake)" -- so it#    is not a genuinely novel held-out task, and step 1 cannot catch it because the rows are not#    exactly equal. `EL_GABY_EXCLUDED_SESSIONS` is the only `mouse_recday ==` special case in#    his notebook.import importlibimport elasticnet_regression_v4 as v4importlib.reload(v4)valid_sessions_dic = {}for mouse_recday in mouse_recdays:    valid_sessions, tasks = [], []    for session in sorted(s for s in data_dic[mouse_recday] if s != 'valid_sessions'):        sd = data_dic[mouse_recday][session]        if sd['num_trials'] < 5:            print(f'{mouse_recday} session {session}: not enough trials, skipping')            continue        if not any(np.array_equal(sd['Task'], c) for c in tasks):            tasks.append(sd['Task'])            valid_sessions.append(session)    valid_sessions_dic[mouse_recday] = valid_sessionsprint('\nEl-Gaby hand-exclusions:')valid_sessions_dic = v4.apply_excluded_sessions(valid_sessions_dic)n_fold = {mr: len(v) for mr, v in valid_sessions_dic.items()}print(f'\nfolds per recday: min {min(n_fold.values())}  median '      f'{int(np.median(list(n_fold.values())))}  max {max(n_fold.values())}')print('recdays with <2 folds (will be skipped):',      [mr for mr, n in n_fold.items() if n < 2] or 'none')

## Run - past and future lagsBoth directions, same config otherwise. `n_jobs` uses joblib's threading backend, so `data_dic` is shared rather than copied to workers.

In [ ]:
import importlib, osfrom datetime import datetimeimport elasticnet_regression_v4 as v4importlib.reload(v4)STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')N_JOBS = 6                      # 8 cores on this box; threading, so memory is shared# 1252 units over 25 recdays, ~6 folds each -> ~20 min per direction at N_JOBS=6.# For El-Gaby's ACTUALLY EXECUTED path set use_poisson=True below: no alpha dropout at# all, but ~1.96 s/fit against ElasticNet's 0.29 s, so budget ~2 h per direction.def make_config(direction, **kw):    # El-Gaby's executed default is Poisson (use_poisson=True, alpha=1, no sparsification)    # but it is ~7x slower per fit. The ElasticNet branch below is the paper's stated one.    return v4.RegressionConfigV4(        use_poisson=False, regularize=True,      # ElasticNet, alpha=0.01, positive        lag_direction=direction,        alpha_mode='fixed',                      # 'relative' + alpha_frac to rescale per neuron        pref_phase_source='train',               # 'test' reproduces the reference's leakage        state_reduce='mean',                     # 'max' also stored either way        **kw)results, pooled, diagnostics = {}, {}, {}for direction in ('past', 'future'):    cfg = make_config(direction)    # Folder name leads with the estimator (poisson / elasticnet / linear) -- it is the    # setting that most changes the numbers. Everything else is in run_config.json,    # written into the same folder as the arrays and figures.    out_dir = os.path.join('../data/figures', v4.run_dir_name(cfg, stamp=STAMP))    print(f'\n{"="*70}\n{direction.upper()} lags -> {out_dir}\n{"="*70}')    results[direction], pooled[direction], diagnostics[direction] = v4.run_and_summarise_all_mice_v4(        data_dic, cfg,        valid_sessions_dic=valid_sessions_dic,        save_dir=out_dir, export_dir=out_dir,        make_pdfs=True, n_jobs=N_JOBS, verbose=True)configs = {d: make_config(d) for d in results}

### Per-recday diagnosticsRead this before the region table. `frac_allzero_fits` is the alpha dropout, `frac_pref_phase_flips` the fraction of neurons whose coordinate frame changes between folds, and `n_nonzero_lag_alt_top3` the mask size with the argsort tie-breaking guard flipped - a large gap there means the mask is partly sort-order artefact.

In [ ]:
for direction, tab in diagnostics.items():    print(f'\n===== {direction} =====')    print(tab.to_string(index=False))

### The state-tuning filter is confounded by leg duration - check this before trusting `selected``selected = nonzero_lag & state_tuned & mean_corr.notna()`, and the `state_tuned` term does notmean what it says. `raw_to_norm` warps each state interval onto 90 bins by *averaging*, so alonger leg puts more raw bins into each normalised bin, lowering its variance and therefore its**max** - which is the statistic the El-Gaby test z-scores across states. Constant-rate Poissoncells with no tuning at all then acquire a "preferred state": the shortest leg.Measured on 300 pure-noise cells per row, nominal alpha = 0.05:| longest:shortest leg | FPR (`max`) | prefer shortest leg | FPR (`mean`) ||---|---|---|---|| 1.0x | 0.053 | 26% (chance) | 0.097 || 2.0x | **0.970** | **94%** | 0.117 || 3.0x | **1.000** | **99%** | 0.073 |The median within-session longest:shortest mean leg duration **in this dataset is 2.26x**(p90 5.4x, max 11.5x, n=164 sessions). So at the real data's leg inequality the filter passesessentially everything and selects on leg geometry, not tuning.And it is measurably present in the real data: across 31 sessions (1,860 units), **47.4% ofreal units "prefer" the shortest leg** against a chance of 25%, and the per-session fractioncorrelates with that session's leg-duration ratio at **r = 0.43**. Real neurons do carrygenuine tuning - the effect is about half the pure-noise strength - but a large share of thepreferred-state assignment is leg geometry.`state_tuning_statistic='mean'` is duration-invariant and stays near nominal. The default stays`'max'` to match the reference - change it deliberately. Both masks are computed on every run(`state_tuned_mask` and `state_tuned_mask_alt`), so the comparison below needs no re-run.

In [ ]:
# How saturated is the tuning filter, and does the preferred state just track the shortest leg?for direction, tab in diagnostics.items():    print(f'\n===== {direction} =====')    print(tab[['mouse_recday', 'n_neurons', 'n_state_tuned', 'n_state_tuned_alt_stat',               'state_duration_ratio', 'frac_pref_state_is_shortest']].to_string(index=False))# The same region table under the duration-invariant statistic. If the anatomy result only# exists under 'max', it is a leg-geometry result.tab = tables['past'].copy()tab['selected'] = tab.nonzero_lag & tab.state_tuned_alt_stat & tab.mean_corr.notna()print('\n===== past lags, state tuning by MEAN (duration-invariant) =====')v4.region_summary(tab)

## Which animals do these neurons come from?PFC has no anatomy, so `build_unit_table` falls back to identity columns (`recday`, `mouse`,`order` = the `Neuron_raw` row index) and groups by animal. Every regression column is the sameone the LEC table carries.

In [ ]:
tables = {d: v4.build_unit_table(results[d], configs[d], data_dic=data_dic)          for d in results}print('has_anatomy:', {d: t.attrs['has_anatomy'] for d, t in tables.items()})for direction, tab in tables.items():    print(f'\n{"="*70}\n{direction.upper()} lags - selection by mouse\n{"="*70}')    v4.region_summary(tab)

### Past vs future`pro_index = (r_future - r_past) / (|r_future| + |r_past|)`: positive means a unit is betterexplained prospectively. The two designs are not degenerate - past lag *k* and future lag 12-*k*point at the same task position one loop apart, and their measured column correlation is only~0.02-0.34, because routes vary between trials.

In [ ]:
merged, direction_summary = v4.compare_directions(tables['past'], tables['future'])merged.head()

### The firing-rate confoundAt `alpha=0.01` a unit is only fittable if it fires fast enough - **57%** of PFC neurons fitall-zero, and the survivors are the fast ones - so a difference in *selection rate* can be adifference in *firing rate*. `region_summary` prints the within-quartile version above. To seehow much is the penalty rather than the biology, re-run one recday with a per-neuron relativealpha.

In [ ]:
# Rate-matched re-run of a single recday (cheap): every neuron sits at the same point on# its own regularization path instead of a shared absolute alpha. 57% of PFC neurons fit# all-zero at the fixed alpha, so this is the check that says how much of the result is the# penalty rather than the biology.mr = list(results['past'].keys())[0]cfg_rel = make_config('past', alpha_mode='relative', alpha_frac=0.1)res_rel = v4.run_cross_validated_regression_v4(    data_dic, mr, cfg_rel, valid_sessions=valid_sessions_dic[mr], verbose=True)tab_rel = v4.build_unit_table({mr: res_rel}, cfg_rel, data_dic=data_dic)print('\n--- relative alpha ---')v4.region_summary(tab_rel)print('\n--- fixed alpha, same recday ---')v4.region_summary(tables['past'][tables['past'].recday == mr])

## Inspecting one neuron`fold_betas` returns each fold's beta matrix collapsed in *that fold's* frame - the un-averaged view of what the `*_foldbetas.pdf` pages show.

In [ ]:
mr = list(results['past'].keys())[0]res = results['past'][mr]top = np.argsort(np.nan_to_num(res['mean_tuning_correlations_pref'], nan=-np.inf))[::-1][:5]print('top neurons by preferred-phase tuning r:', top.tolist())ni = int(top[0])print(f'neuron {ni}: pref phase per fold = {res["pref_phases"][ni].tolist()}, '      f'peak lag = {res["peak_lags"][ni]}, r = {res["mean_corrs"][ni]:.3f}, '      f'non-zero betas/fold = {res["n_nonzero_betas"][ni].tolist()}')fb = v4.fold_betas(res, configs['past'], ni)          # (n_folds, 9 locations, 12 lags)fig, axes = plt.subplots(1, len(fb) + 1, figsize=(2.2 * (len(fb) + 1), 2.4))vmax = np.nanmax(np.abs(fb)) or 1.0for fi, ax in enumerate(axes[:-1]):    ax.imshow(fb[fi], aspect='auto', cmap='hot', vmin=0, vmax=vmax)    ax.set_title(f'fold {fi} (pref {res["pref_phases"][ni, fi]})', fontsize=7)    ax.tick_params(labelsize=5)B, modal, n_used, n_tot = v4.betas_in_common_frame(res, configs['past'], ni)axes[-1].imshow(B, aspect='auto', cmap='hot', vmin=0, vmax=vmax)axes[-1].set_title(f'mean [{n_used}/{n_tot} folds, pref {modal}]', fontsize=7)axes[-1].tick_params(labelsize=5)fig.suptitle(f'{mr} neuron {ni} - past lags (location x lag)', fontsize=9)fig.tight_layout()

## Reading the exports backOne `.npz` per recday per direction, plus six PDFs:| file | what it shows ||---|---|| `*_all.pdf` / `*_nonzerolag.pdf` | summary page per neuron: betas, 360-bin curves, n=4 readout || `*_all_foldbetas.pdf` / `*_nonzerolag_foldbetas.pdf` | beta matrix **per fold**, each in its own frame || `*_all_foldratemaps.pdf` / `*_nonzerolag_foldratemaps.pdf` | actual vs predicted **per fold** |The per-fold rate maps are the honest view: each fold holds out a *different task*, so theactual tuning curve genuinely differs between columns, and the summary page's fold-averagedactual blends them. Note also that a single fold's `r` comes from 4 points, whose null samplingSD is 1/sqrt(3) = 0.58 - read the shapes, not the individual numbers.Nothing below needs `data_dic`.

In [ ]:
import globout_dir = os.path.join('../data/figures',                       v4.run_dir_name(configs['past'], stamp=STAMP))print('\n'.join(sorted(os.path.basename(p) for p in glob.glob(out_dir + '/*'))[:12]))z = v4.load_regression_outputs(glob.glob(out_dir + '/*_arrays.npz')[0])for k in sorted(z):    print(f'  {k:32s} {getattr(z[k], "shape", type(z[k]).__name__)}')

## LEC vs PFC[`elasticnet_v4_compare.py`](../../code/elasticnet_v4_compare.py) reads the exported `.npz`files from both datasets - no `data_dic`, no re-fit. Needs the LEC sweep to have been run too.The unit of analysis is a **mouse_recday** (mean +/- SEM across recdays), and `min_neurons=2`drops degenerate recdays - `me10_20122021_21122021` has exactly one neuron.Read the diagnostics table alongside the result. The selection criterion carries a largebaseline on both datasets: the non-zero-lag test fires on ~22% of pure Poisson noise, and thestate-tuning filter is confounded by leg duration. A dataset difference in selection rate isonly interpretable against those.

In [ ]:
import syssys.path.insert(0, '../../code')import elasticnet_v4_compare as cmpimportlib.reload(cmp)# ESTIMATOR pins which run to compare -- folders are named {estimator}_v4_{direction}_{stamp},# and compare_datasets also reads each run_config.json and warns if the settings differ.ESTIMATOR = v4.estimator_name(configs['past'])dirs = {ds: {d: cmp.resolve_latest(root, d, ESTIMATOR) for d in ('past', 'future')}        for ds, root in cmp.DEFAULT_ROOTS.items()}for ds, by_dir in dirs.items():    for d, path in by_dir.items():        print(f'{ds} {d}: {path}')stats, table, fig = cmp.compare_datasets(    dirs, min_neurons=2,    out_path=f'../data/figures/{ESTIMATOR}_v4_lec_vs_pfc')